In [18]:
from retrieval.orchestration import IngestionService
from retrieval.embeddings import OpenAIEmbedder, HuggingFaceEmbedder
from openai import OpenAI
from retrieval.database import DataBase
from retrieval.vector_storage import VectorStorage
from qdrant_client import QdrantClient
from settings.config import Config
config = Config()
openai_client = OpenAI(api_key=config.open_api_key)
qdrant_client = QdrantClient(url=config.qdrant_url, api_key=config.qdrant_api_key, timeout=1000)

INFO:httpx:HTTP Request: GET https://a006ec8b-3148-417e-a19b-ce5c09338b16.us-east4-0.gcp.cloud.qdrant.io:6333 "HTTP/1.1 200 OK"


In [19]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance
import logging
logger = logging.getLogger(__name__)

class CollectionManager():
    def __init__(self, qdrant_client: QdrantClient):
        self.qdrant_client = qdrant_client
        logger.info("CollectionManager initialized with QdrantClient.")
        
    def create_collection(self, collection_name: str, vector_size: int, distance: Distance):
        vector_params = VectorParams(size=vector_size, distance=distance)

        if not self.qdrant_client.collection_exists(collection_name):
            try:
                response = self.qdrant_client.create_collection(collection_name=collection_name, vectors_config=vector_params)
                if response:
                    logger.info(f"Collection '{collection_name}' created successfully with vector size {vector_size} and distance '{distance}'.")
                    return response
                else:
                    logger.warning(f"Failed to create collection '{collection_name}'.")
                    return response
            except Exception as e:
                logger.error(f"Error creating collection: {e}")
                raise
        else:
            logger.warning(f"Collection '{collection_name}' already exists. Use replace_collection() instead.")
            return None
        
        
    def replace_collection(self, collection_name: str, vector_size: int, distance: Distance):
        logger.info(f"Replacing collection '{collection_name}' with new configuration.")
        self.delete_collection(collection_name)
        return self.create_collection(collection_name, vector_size, distance)

    def delete_collection(self, collection_name: str):
        try:
            response = self.qdrant_client.delete_collection(collection_name=collection_name)
            if response:
                logger.info(f"Collection '{collection_name}' deleted successfully.")
                return response
            else:
                logger.warning(f"Failed to delete collection '{collection_name}'.")
                return response
            
        except Exception as e:
            logger.error(f"Error deleting collection: {e}")
            raise

    def get_collections(self):
        try:
            response = self.qdrant_client.get_collections()
            collections = [collection.name for collection in response.collections]
            if collections:
                logger.info(f"Retrieved collections: {collections}")
                return collections
            else:
                logger.warning("No collections found.")
                return []
        except Exception as e:
            logger.error(f"Error retrieving collections: {e}")